# 01 — Data Load

**Purpose.** Load the SemEval-2018 Task 1 (E-c) English corpus, standardise the
column names, normalise the emotion labels, run integrity checks, and write a
clean interim copy that every later notebook reads.

| | |
|---|---|
| **Input**  | `SemEval2018-Task1-{train,dev,test}.txt` (tab separated) |
| **Output** | `data/interim/{train,dev,test}_raw.csv` |
| **Next**   | `02_eda.ipynb` |

Nothing here touches the text itself — no cleaning, no emoji handling. That
happens in `03_model_train.ipynb`, so that `02_eda.ipynb` can profile the corpus
exactly as it arrived.

In [1]:
import os
from pathlib import Path

ON_KAGGLE = os.path.exists("/kaggle/working")

if ON_KAGGLE:
    PROJECT = Path("/kaggle/working")
    RAW_CANDIDATES = [Path("/kaggle/input")]
else:
    # notebooks/ lives one level below the project root
    PROJECT = Path.cwd()
    if PROJECT.name == "notebooks":
        PROJECT = PROJECT.parent
    RAW_CANDIDATES = [PROJECT / "data" / "raw"]

INTERIM   = PROJECT / "data" / "interim"
PROCESSED = PROJECT / "data" / "processed"
MODELS    = PROJECT / "models"
RESULTS   = PROJECT / "results"
FIGURES   = PROJECT / "figures"
for d in (INTERIM, PROCESSED, MODELS, RESULTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

EMOTION_LABELS = [
    "anger", "anticipation", "disgust", "fear", "joy", "love",
    "optimism", "pessimism", "sadness", "surprise", "trust",
]
TRACKS = [
    ("no_emoji",   "text_no_emoji",   "Without Emoji"),
    ("with_emoji", "text_with_emoji", "With Emoji"),
]

def display_path(p):
    """Print paths relative to PROJECT so machine-specific parents never leak."""
    if p is None:
        return None
    p = Path(p)
    try:
        return str(p.resolve().relative_to(PROJECT.resolve()))
    except ValueError:
        return p.name

print("project root :", PROJECT.name)
print("on kaggle    :", ON_KAGGLE)

project root : krish
on kaggle    : False


## 1.1 Locate the source files

In [2]:
import pandas as pd

FILES = {
    "train": "SemEval2018-Task1-train.txt",
    "dev":   "SemEval2018-Task1-dev.txt",
    "test":  "SemEval2018-Task1-test.txt",
}


def find_file(filename):
    """Search the candidate roots recursively; Kaggle nests dataset folders."""
    for root in RAW_CANDIDATES:
        if not root.exists():
            continue
        for path in root.rglob(filename):
            return path
    return None


paths = {}
for split, fname in FILES.items():
    p = find_file(fname)
    paths[split] = p
    print(f"{split:6s} -> {display_path(p) if p else 'NOT FOUND'}")

missing = [s for s, p in paths.items() if p is None]
if missing:
    raise FileNotFoundError(
        f"Could not find: {missing}\n"
        f"Searched: {[str(r) for r in RAW_CANDIDATES]}\n"
        "Locally, put the three .txt files in data/raw/. "
        "On Kaggle, attach the SemEval-2018 Task 1 dataset to the notebook."
    )

train  -> data/raw/SemEval2018-Task1-train.txt
dev    -> data/raw/SemEval2018-Task1-dev.txt
test   -> data/raw/SemEval2018-Task1-test.txt


## 1.2 Load

In [3]:
raw = {split: pd.read_csv(p, sep="\t") for split, p in paths.items()}

for split, df in raw.items():
    print(f"{split:6s} shape={df.shape}")

print("\ncolumns as loaded:")
print(list(raw["train"].columns))
raw["train"].head(3)

train  shape=(6838, 13)
dev    shape=(886, 13)
test   shape=(3259, 13)

columns as loaded:
['ID', 'Tweet', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']


,ID,Tweet,anger,anticipation,disgust,fear,joy,love,optimism,pessimism,sadness,surprise,trust
0,2017-En-21441,“Worry is a down payment on a problem you may ...,0,1,0,0,0,0,1,0,0,0,1
1,2017-En-31535,Whatever you decide to do make sure it makes y...,0,0,0,0,1,1,1,0,0,0,0
2,2017-En-21068,@Max_Kellerman it also helps that the majorit...,1,0,1,0,1,0,1,0,0,0,0


## 1.3 Standardise column names

The released files use `ID` / `Tweet` plus capitalised emotion columns. Lower-casing
everything and renaming `Tweet` to `text` gives one stable schema for the rest of
the pipeline.

In [4]:
for split, df in raw.items():
    df.columns = [c.lower() for c in df.columns]
    df.rename(columns={"tweet": "text"}, inplace=True)

print("columns after standardisation:")
print(list(raw["train"].columns))

# every expected emotion column must be present
for split, df in raw.items():
    absent = set(EMOTION_LABELS) - set(df.columns)
    assert not absent, f"{split} is missing emotion columns: {sorted(absent)}"
print("\nall 11 emotion columns present in all three splits")

columns after standardisation:
['id', 'text', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'love', 'optimism', 'pessimism', 'sadness', 'surprise', 'trust']

all 11 emotion columns present in all three splits


## 1.4 Normalise the emotion labels

Some rows carry the string `NONE` instead of `0`. Mapping that to zero and casting
to integer makes the label block a clean binary matrix.

In [5]:
for split, df in raw.items():
    for col in EMOTION_LABELS:
        df[col] = df[col].replace("NONE", 0)
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

# after this every label cell must be exactly 0 or 1
for split, df in raw.items():
    vals = pd.unique(df[EMOTION_LABELS].values.ravel())
    assert set(vals) <= {0, 1}, f"{split} has unexpected label values: {vals}"
print("all label values are binary in all three splits")

raw["train"][["text"] + EMOTION_LABELS[:4]].head(3)

all label values are binary in all three splits


,text,anger,anticipation,disgust,fear
0,“Worry is a down payment on a problem you may ...,0,1,0,0
1,Whatever you decide to do make sure it makes y...,0,0,0,0
2,@Max_Kellerman it also helps that the majorit...,1,0,1,0


## 1.5 Integrity checks

In [6]:
report = []
for split, df in raw.items():
    report.append({
        "split": split,
        "rows": len(df),
        "null text": int(df["text"].isna().sum()),
        "empty text": int((df["text"].astype(str).str.strip() == "").sum()),
        "duplicate ids": int(df["id"].duplicated().sum()) if "id" in df else None,
        "duplicate text": int(df["text"].duplicated().sum()),
        "rows with no label": int((df[EMOTION_LABELS].sum(axis=1) == 0).sum()),
        "labels per tweet (mean)": round(df[EMOTION_LABELS].sum(axis=1).mean(), 2),
    })

checks = pd.DataFrame(report)
display(checks)

assert checks["null text"].sum() == 0, "null text found"
assert checks["empty text"].sum() == 0, "empty text found"
print("no null or empty text anywhere")

,split,rows,null text,empty text,duplicate ids,duplicate text,rows with no label,labels per tweet (mean)
0,train,6838,0,0,0,0,204,2.35
1,dev,886,0,0,0,0,14,2.44
2,test,3259,0,0,0,0,75,2.41


no null or empty text anywhere


Rows carrying no positive label at all are legitimate in this corpus — an
annotator may judge a tweet to convey none of the eleven emotions — so they are
kept rather than dropped.

## 1.6 Save the interim copy

In [7]:
KEEP = (["id"] if "id" in raw["train"].columns else []) + ["text"] + EMOTION_LABELS

for split, df in raw.items():
    out = INTERIM / f"{split}_raw.csv"
    df[KEEP].to_csv(out, index=False, encoding="utf-8")
    print(f"saved {display_path(out)}  ({len(df)} rows)")

print("\ninterim directory now contains:")
for p in sorted(INTERIM.iterdir()):
    print(" ", p.name, f"{p.stat().st_size/1024:.0f} KB")

saved data/interim/train_raw.csv  (6838 rows)
saved data/interim/dev_raw.csv  (886 rows)
saved data/interim/test_raw.csv  (3259 rows)

interim directory now contains:
  dev_raw.csv 115 KB
  test_raw.csv 424 KB
  train_raw.csv 892 KB


## Summary

The three splits are loaded, the schema is standardised, the labels are a clean
binary matrix, and the integrity checks pass. The corpus is untouched otherwise,
ready to be profiled in `02_eda.ipynb`.